# BUSINESS LOGIC & DATA ENGINEERING

### Step 0: Load the needed information 

In [0]:
%run "./0.0 Utils"

In [0]:
from pyspark.sql.functions import col, countDistinct

directory_bronze = f"{directory_notebook}/data"

# Load datasets
characters = spark.read.csv(directory_bronze + "/characters.csv", header=True, inferSchema=True)
starships = spark.read.csv(directory_bronze + "/starships.csv", header=True, inferSchema=True)
species = spark.read.csv(directory_bronze + "/species.csv", header=True, inferSchema=True)
cities = spark.read.csv(directory_bronze + "/cities.csv", header=True, inferSchema=True)
organizations = spark.read.csv(directory_bronze + "/organizations.csv", header=True, inferSchema=True)
battles = spark.read.csv(directory_bronze + "/battles.csv", header=True, inferSchema=True)

### Character who lived the longest
- Calculate the lifespan by subtracting birth year from death year
- Sort descending to find the top result

In [0]:
# Character who lived the longest
characters_lifespan = characters.withColumn("lifespan", col("year_died") - col("year_born"))
longest_lived = characters_lifespan.orderBy(col("lifespan").desc()).select("name", "lifespan").first()
print(f"Character who lived the longest: {longest_lived['name']} ({longest_lived['lifespan']} years)")

### Number of distinct starships
- Use countDistinct to see how many unique starship names exist

In [0]:
# Number of distinct starships
num_starships = starships.select(countDistinct("name")).first()[0]
print(f"Number of distinct starships: {num_starships}")

### Species with the highest average lifespan
- Sort species by the "average_lifespan" column

In [0]:
# Species with the highest life expectancy
species_life = species.select("name", "average_lifespan").orderBy(col("average_lifespan").desc()).filter(col("average_lifespan").isNotNull()).first()
print(f"Species with the highest life expectancy: {species_life['name']} ({species_life['average_lifespan']} years)")

### City with the lowest population
- Sort cities by population ascending to find the least populated

In [0]:
# City with the lowest population
least_populated_city = cities.orderBy(col("population").asc()).select("name", "population").filter(col("population").isNotNull()).first()
print(f"City with the lowest population: {least_populated_city['name']} ({least_populated_city['population']} inhabitants)")

### Sort cities by population
- Get a full list of cities ordered by population descending

In [0]:
# Order cities by population
sorted_cities = cities.orderBy(col("population").desc())
print("\nCities ordered by population:")
display(sorted_cities.select("name", "population"))

### Organization with the longest duration
- Calculate the difference between founding and dissolution years
- Sort descending to find the top organization

In [0]:
# Organization with the longest duration
orgs_duration = organizations.withColumn(
    "duration", 
    col("dissolved") - col("founded")
)
longest_org = orgs_duration.orderBy(
    col("duration").desc()
).select("name", "duration").first()
print(f"Organization that lasted the longest: {longest_org['name']} ({longest_org['duration']} years)")

### Leader of the organization involved in the most battles
- Explode the list of battle participants, count how many times each org appears
- Join with organization data to match leaders, and sort descending

In [0]:
# Leader of the organization involved in the most battles
from pyspark.sql.functions import explode, split

# Prepare participants column (split comma-separated organizations)
battles_clean = battles.withColumn("org", explode(split(col("participants"), ", ")))

# Count battles per organization
org_battle_count = battles_clean.groupBy("org").count()

# Join with organizations to get leader
orgs_with_leaders = organizations.select(col("name").alias("org"), "leader")
leader_battles = org_battle_count.join(orgs_with_leaders, on="org", how="inner").orderBy(col("count").desc())

# Show top leader
top_leader = leader_battles.select("leader", "count").first()
print(f"Leader who participated in the most battles: {top_leader['leader']} ({top_leader['count']} battles)")

---

# Now it's your turn!
### Answering the following questions using the same techniques above.



### 1) How many unique planets are listed in the dataset?


In [0]:
#TODO

### 2) What is the average height of all characters?


In [0]:
#TODO

### 3) Which gender appears most frequently among the characters?


In [0]:
#TODO

### 4) Which species appears in the highest number of records among the characters?

write the top-3 results in a parquet table in path `f"file:{os.getcwd()}/output/common_species"` 


In [0]:
#TODO

### 5) What is the most common eye color among species with a average lifespan over 100?
Write the results in two formats:

- as a parquet table on you local workspace folder. Table must be partioned by "eye_colors". Write the output in `f"file:{os.getcwd()}/output/parquet_output"`
- as avro format with 3 partitions. Write the output in `f"file:{os.getcwd()}/output/avro_output"`

In [0]:
#TODO

# Congratulations!
## You've completed the business logic and data engineering notebook!
Now, continue with the next notebook about ML.

---